# 22c - Ablation Study: LSTM on WLASL-500

**Ablation Study** — identical training setup as Notebook 17e, but replacing
the GRU recurrent layer with an LSTM.

## Architecture: PoseClassifierLSTM
- Input projection: Linear → LayerNorm → ReLU → Dropout
- **LSTM** (bidirectional, 2 layers, hidden=256) — replaces GRU
- Temporal Attention (unchanged from Nb 17x)
- Classifier head (unchanged from Nb 17x)

## How to run
Run all cells in order. Expected Top-1 for 500 classes: ~20-30%.

In [ ]:
# === Imports and Configuration ===
# ⬇️  Only this line changes between notebooks 22a / 22b / 22c / ...
SUBSET_PATH = "/home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/subsets/wlasl_500_subset.json"
# ════════════════════════════════════════════════════════════════

import numpy as np
import json
import random
import time
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SUBSET_PATH = Path(SUBSET_PATH)
LANDMARKS_ROOT = Path("/home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl2000")
EXPERIMENTS_BASE = Path("/home/fransando/Dev/tfg-sign-language-recognition/05_experiments")

with open(SUBSET_PATH) as f:
    subset_meta = json.load(f)

EXPERIMENT_NAME = subset_meta['name']
NUM_CLASSES = subset_meta['num_classes']
OUTPUT_DIR = EXPERIMENTS_BASE / f"{EXPERIMENT_NAME}_lstm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if NUM_CLASSES >= 800:
    _epochs   = 250
    _patience = 50
else:
    _epochs   = 150
    _patience = 30

CONFIG = {
    'max_frames':      64,
    'input_size':      63,
    'num_classes':     NUM_CLASSES,
    'hidden_size':     256,
    'num_layers':      2,
    'dropout':         0.4,
    'bidirectional':   True,
    'batch_size':      32,
    'epochs':          _epochs,
    'lr':              1e-3,
    'weight_decay':    5e-4,
    'label_smoothing': 0.1,
    'patience':        _patience,
}

print(f'Date:        {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Device:      {device}')
print(f'Subset:      {EXPERIMENT_NAME}')
print(f'Classes:     {NUM_CLASSES}')
print(f'Output dir:  {OUTPUT_DIR}')
print(f'Epochs:      {CONFIG["epochs"]}  (patience={CONFIG["patience"]})')
print(f'')
stats = subset_meta['statistics']
print(f'Dataset stats:')
print(f'  Train: {stats["train"]}')
print(f'  Val:   {stats["val"]}')
print(f'  Test:  {stats["test"]}')
print(f'  Total: {stats["total"]}')
print(f'  Mean videos/class: {stats["mean_videos_per_class"]}')
print(f'  Class imbalance (CV): {stats["cv_pct"]}%')
print(f'  Classes with no val:  {stats["classes_with_no_val"]}')

In [ ]:
# === Dataset and DataLoaders ===

class SkeletalAugmentation:
    def __init__(self, rotation=10, scale=(0.9, 1.1), translation=0.1, noise=0.01, p=0.5):
        self.rotation = rotation; self.scale = scale
        self.translation = translation; self.noise = noise; self.p = p

    def __call__(self, landmarks):
        landmarks = landmarks.copy()
        if random.random() < self.p:
            angle = np.radians(random.uniform(-self.rotation, self.rotation))
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            x, y = landmarks[:, :, 0].copy(), landmarks[:, :, 1].copy()
            landmarks[:, :, 0] = x * cos_a - y * sin_a
            landmarks[:, :, 1] = x * sin_a + y * cos_a
        if random.random() < self.p: landmarks *= random.uniform(*self.scale)
        if random.random() < self.p:
            landmarks[:, :, 0] += random.uniform(-self.translation, self.translation)
            landmarks[:, :, 1] += random.uniform(-self.translation, self.translation)
        if random.random() < self.p:
            landmarks += np.random.normal(0, self.noise, landmarks.shape)
        return landmarks


class SubsetDataset(Dataset):
    def __init__(self, samples, landmarks_root, max_frames=64, augmentation=None):
        self.samples = samples; self.root = landmarks_root
        self.max_frames = max_frames; self.augmentation = augmentation

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        landmarks = np.load(self.root / item['path'])
        landmarks = self._pad_or_truncate(landmarks)
        if self.augmentation: landmarks = self.augmentation(landmarks)
        return torch.tensor(landmarks.reshape(self.max_frames, -1), dtype=torch.float32), item['gloss_idx']

    def _pad_or_truncate(self, lm):
        T = lm.shape[0]
        if T > self.max_frames: return lm[np.linspace(0, T-1, self.max_frames, dtype=int)]
        if T < self.max_frames: return np.concatenate([lm, np.zeros((self.max_frames-T, 21, 3))], axis=0)
        return lm


aug = SkeletalAugmentation()
train_ds = SubsetDataset(subset_meta['samples']['train'], LANDMARKS_ROOT, CONFIG['max_frames'], augmentation=aug)
val_ds   = SubsetDataset(subset_meta['samples']['val'],   LANDMARKS_ROOT, CONFIG['max_frames'])
test_ds  = SubsetDataset(subset_meta['samples']['test'],  LANDMARKS_ROOT, CONFIG['max_frames'])
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
print(f'Train: {len(train_ds):>6} samples  ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds):>6} samples  ({len(val_loader)} batches)')
print(f'Test:  {len(test_ds):>6} samples  ({len(test_loader)} batches)')
all_labels = [item['gloss_idx'] for item in subset_meta['samples']['train']]
print(f'Label range: [{min(all_labels)}, {max(all_labels)}] | Unique: {len(set(all_labels))} / {NUM_CLASSES}')

In [ ]:
# === Model: PoseClassifierLSTM ===

class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.Tanh(),
            nn.Linear(hidden_size // 2, 1, bias=False),
        )
    def forward(self, rnn_out):
        weights = F.softmax(self.attention(rnn_out).squeeze(-1), dim=1)
        return torch.bmm(weights.unsqueeze(1), rnn_out).squeeze(1), weights


class PoseClassifierLSTM(nn.Module):
    def __init__(self, config):
        super().__init__()
        hs  = config['hidden_size']; bidir = config.get('bidirectional', True)
        rhs = hs * (2 if bidir else 1)
        self.input_projection = nn.Sequential(
            nn.Linear(config['input_size'], hs), nn.LayerNorm(hs), nn.ReLU(), nn.Dropout(config['dropout']))
        self.lstm = nn.LSTM(input_size=hs, hidden_size=hs, num_layers=config['num_layers'],
            batch_first=True, dropout=config['dropout'] if config['num_layers'] > 1 else 0, bidirectional=bidir)
        self.attention = TemporalAttention(rhs)
        self.classifier = nn.Sequential(
            nn.Dropout(config['dropout']), nn.Linear(rhs, hs), nn.ReLU(),
            nn.Dropout(config['dropout']), nn.Linear(hs, config['num_classes']))

    def forward(self, x, return_attention=False):
        x = self.input_projection(x)
        rnn_out, _ = self.lstm(x)  # _ discards (h_n, c_n)
        context, attn_w = self.attention(rnn_out)
        logits = self.classifier(context)
        return (logits, attn_w) if return_attention else logits


model = PoseClassifierLSTM(CONFIG).to(device)
print(f'Model: PoseClassifierLSTM | Parameters: {sum(p.numel() for p in model.parameters()):,} | Classes: {NUM_CLASSES}')

In [ ]:
# === Training and Evaluation Functions ===

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x); loss = criterion(out, y); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += out.argmax(1).eq(y).sum().item(); total += x.size(0)
    return total_loss / total, 100 * correct / total

@torch.no_grad()
def evaluate(model, loader, top_k=(1, 5)):
    model.eval()
    total = 0; topk_correct = {k: 0 for k in top_k}; all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device); logits = model(x); total += y.size(0)
        _, pred_topk = logits.topk(max(top_k), 1, True, True)
        for k in top_k:
            topk_correct[k] += pred_topk[:, :k].eq(y.view(-1,1).expand_as(pred_topk[:,:k])).any(dim=1).sum().item()
        all_preds.extend(logits.argmax(1).cpu().numpy()); all_labels.extend(y.cpu().numpy())
    return {f'top{k}_acc': 100.0*topk_correct[k]/total for k in top_k}, np.array(all_preds), np.array(all_labels)

print('Training functions defined.')

In [ ]:
# === Training Loop ===

criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
history = {'train_loss': [], 'train_acc': [], 'val_top1': [], 'val_top5': []}
best_val_top1 = 0.0; best_epoch = 0; patience_counter = 0
train_start = time.time(); CHECKPOINT_PATH = OUTPUT_DIR / 'best_model.pt'

print(f'Training {EXPERIMENT_NAME}_lstm for up to {CONFIG["epochs"]} epochs (patience={CONFIG["patience"]})')
print('=' * 65)

for epoch in range(CONFIG['epochs']):
    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_metrics, _, _ = evaluate(model, val_loader)
    scheduler.step()
    history['train_loss'].append(train_loss); history['train_acc'].append(train_acc)
    history['val_top1'].append(val_metrics['top1_acc']); history['val_top5'].append(val_metrics['top5_acc'])
    if val_metrics['top1_acc'] > best_val_top1:
        best_val_top1 = val_metrics['top1_acc']; best_epoch = epoch + 1; patience_counter = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_top1': best_val_top1,
                    'config': CONFIG, 'subset': EXPERIMENT_NAME, 'architecture': 'PoseClassifierLSTM'}, CHECKPOINT_PATH)
    else:
        patience_counter += 1
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Ep {epoch+1:3d}/{CONFIG["epochs"]} | Loss {train_loss:.3f} | Train {train_acc:.1f}% | '
              f'Val {val_metrics["top1_acc"]:.1f}% (Top-5 {val_metrics["top5_acc"]:.1f}%) | '
              f'Best {best_val_top1:.1f}% | {time.time()-t0:.1f}s')
    if patience_counter >= CONFIG['patience']:
        print(f'\nEarly stopping at epoch {epoch + 1}'); break

elapsed = time.time() - train_start
print(f'\nTraining finished in {elapsed/60:.1f} min | Best Val Top-1: {best_val_top1:.1f}% (epoch {best_epoch})')

In [ ]:
# === Training Curves ===
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_ran = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs_ran, history['train_loss'], color='steelblue')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_ran, history['train_acc'], label='Train', color='steelblue')
axes[1].plot(epochs_ran, history['val_top1'], label='Val Top-1', color='darkorange')
axes[1].axhline(best_val_top1, color='red', linestyle='--', label=f'Best: {best_val_top1:.1f}%')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].plot(epochs_ran, history['val_top5'], color='green', label='Val Top-5')
axes[2].plot(epochs_ran, history['val_top1'], color='darkorange', label='Val Top-1')
axes[2].set_title('Val Top-1 vs Top-5'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle(f'{EXPERIMENT_NAME}_lstm — Training Curves (PoseClassifierLSTM)', fontsize=13)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# === Final Evaluation ===
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best checkpoint (epoch {ckpt["epoch"]+1}, val_top1={ckpt["val_top1"]:.1f}%)')
val_metrics,  val_preds,  val_labels  = evaluate(model, val_loader)
test_metrics, test_preds, test_labels = evaluate(model, test_loader)
print(f'\nRESULTS: {EXPERIMENT_NAME}_lstm')
print(f'Val  — Top-1: {val_metrics["top1_acc"]:.1f}%  Top-5: {val_metrics["top5_acc"]:.1f}%  (n={len(val_ds)})')
print(f'Test — Top-1: {test_metrics["top1_acc"]:.1f}%  Top-5: {test_metrics["top5_acc"]:.1f}%  (n={len(test_ds)})')

In [ ]:
# === Per-Class Accuracy + Top-20 Confused Pairs ===
idx_to_gloss = subset_meta['idx_to_gloss']
per_class_acc = {}
for cls_idx in range(NUM_CLASSES):
    mask = test_labels == cls_idx
    if mask.sum() == 0: continue
    per_class_acc[cls_idx] = {'gloss': idx_to_gloss[str(cls_idx)],
                              'acc': float(100.0*(test_preds[mask]==cls_idx).sum()/mask.sum()),
                              'n_test': int(mask.sum())}
accs = [v['acc'] for v in per_class_acc.values()]
print(f'Per-class — Mean: {np.mean(accs):.1f}%  Std: {np.std(accs):.1f}%  100%: {sum(1 for a in accs if a==100)}  0%: {sum(1 for a in accs if a==0)}')
wrong_mask = test_preds != test_labels
pair_counts = Counter(zip(test_labels[wrong_mask].tolist(), test_preds[wrong_mask].tolist()))
print(f'\nTop-20 confused pairs:')
for (t, p), c in pair_counts.most_common(20):
    print(f'  {idx_to_gloss.get(str(t),str(t)):<22} → {idx_to_gloss.get(str(p),str(p)):<22} ({c}x)')

In [ ]:
# === Save Results JSON ===
results = {
    'experiment': f'{EXPERIMENT_NAME}_lstm', 'architecture': 'PoseClassifierLSTM',
    'date': datetime.now().strftime('%Y-%m-%d %H:%M'), 'subset_path': str(SUBSET_PATH),
    'num_classes': NUM_CLASSES, 'config': CONFIG,
    'dataset': {'train': len(train_ds), 'val': len(val_ds), 'test': len(test_ds)},
    'training': {'best_val_top1': best_val_top1, 'best_epoch': best_epoch,
                 'total_epochs': len(history['train_acc']), 'training_time_s': round(elapsed, 1)},
    'val_metrics': val_metrics, 'test_metrics': test_metrics,
    'per_class_accuracy': {str(k): v for k, v in per_class_acc.items()},
    'history': {k: [round(float(v), 4) for v in vals] for k, vals in history.items()},
}
with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'Saved: {OUTPUT_DIR / "results.json"}')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size//1024} KB)')